## Data Preprocessing for Weather and Rainfall Data (Fixed)

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
rainfall_df = pd.read_csv('../data/district_wise_rainfall_normal.csv')
weather_df  = pd.read_csv('../data/weather-1.csv')

In [ ]:
print('Rainfall dataset shape:', rainfall_df.shape)
print('Weather Dataset shape: ', weather_df.shape)
print('\nRainfall Columns:', rainfall_df.columns.tolist())
print('\nWeather Columns: ', weather_df.columns.tolist())

In [ ]:
# Select required columns
rainfall_df = rainfall_df[['STATE_UT_NAME', 'DISTRICT', 'ANNUAL']].copy()
weather_df  = weather_df[['State/UT', 'District', 'Temperature (°C)', 'Humidity (%)']].copy()

In [ ]:
# Rename columns
rainfall_df.rename(columns={
    'STATE_UT_NAME': 'state',
    'DISTRICT':      'district',
    'ANNUAL':        'average_annual_rainfall'
}, inplace=True)

weather_df.rename(columns={
    'State/UT':         'state',
    'District':         'district',
    'Temperature (°C)': 'average_temperature',
    'Humidity (%)':     'average_humidity'
}, inplace=True)

In [ ]:
# Normalize join keys: uppercase + strip whitespace
for df in [rainfall_df, weather_df]:
    for col in ['state', 'district']:
        df[col] = df[col].astype(str).str.upper().str.strip()

In [ ]:
# FIX 1: Convert numeric columns correctly — pass the column, not a scalar
rainfall_df['average_annual_rainfall'] = pd.to_numeric(
    rainfall_df['average_annual_rainfall'], errors='coerce'
)
weather_df['average_temperature'] = pd.to_numeric(
    weather_df['average_temperature'], errors='coerce'
)
weather_df['average_humidity'] = pd.to_numeric(
    weather_df['average_humidity'], errors='coerce'
)

In [ ]:
# Deduplicate weather to one reading per district
weather_deduped = weather_df.drop_duplicates(subset=['state', 'district'], keep='first')

# Compute state-level averages as fallback for unmatched districts
state_avg = (
    weather_df
    .groupby('state')[['average_temperature', 'average_humidity']]
    .mean()
    .reset_index()
    .rename(columns={
        'average_temperature': 'state_avg_temp',
        'average_humidity':    'state_avg_hum'
    })
)

In [ ]:
# FIX 2: Use LEFT join so all 641 rainfall districts are preserved.
# Original inner join silently dropped ~480 districts that had no exact
# name match in weather_df (only 286 of 637 districts matched).
final_df = pd.merge(
    rainfall_df,
    weather_deduped[['state', 'district', 'average_temperature', 'average_humidity']],
    on=['state', 'district'],
    how='left'
)

# Attach state-level fallback values
final_df = pd.merge(final_df, state_avg, on='state', how='left')

In [ ]:
# FIX 3: Fill NaN using assignment (avoids pandas Copy-on-Write ChainedAssignmentError)
# Priority: district match -> state average -> global mean
final_df['average_temperature'] = final_df['average_temperature'].fillna(final_df['state_avg_temp'])
final_df['average_humidity']    = final_df['average_humidity'].fillna(final_df['state_avg_hum'])

final_df.drop(columns=['state_avg_temp', 'state_avg_hum'], inplace=True)

# Global mean fallback for any states with zero weather coverage
final_df['average_temperature']     = final_df['average_temperature'].fillna(final_df['average_temperature'].mean())
final_df['average_humidity']        = final_df['average_humidity'].fillna(final_df['average_humidity'].mean())
final_df['average_annual_rainfall'] = final_df['average_annual_rainfall'].fillna(final_df['average_annual_rainfall'].mean())

In [ ]:
final_df.drop_duplicates(inplace=True)
final_df.reset_index(drop=True, inplace=True)

In [ ]:
print('Final dataset shape:', final_df.shape)
print('Columns:', final_df.columns.tolist())
print('\nNull counts:')
print(final_df.isnull().sum())
print('\nSample:')
print(final_df.head())
print('\nDescribe:')
print(final_df.describe())

In [ ]:
final_df.to_csv('../data/final_environment_dataset.csv', index=False)
print('Dataset saved successfully!')